<h1>Interpretabilidad XAI &mdash; exp08_ordinal_sord_qwk_descongelado</h1>
<p>Subsistema de interpretabilidad (XAI) para el experimento canonico de la tesis.</p>
<p>Organizado en seis pasos:</p>
<ol>
  <li><b>Paso 1a</b>: Carga del modelo clasificador exp08 (MammoVLM)</li>
  <li><b>Paso 1b</b>: Carga del pipeline RAG (FAISS + PubMedBERT + Qwen2.5-7B)</li>
  <li><b>Paso 2</b>: Atribucion del clasificador (IG y Grad-CAM por cabeza)</li>
  <li><b>Paso 3</b>: Metricas del clasificador (Deletion AUC, Pointing Game, IoU)</li>
  <li><b>Paso 4</b>: Atribucion del RAG (Shapley exacto + grounding NLI)</li>
  <li><b>Paso 5</b>: Metricas del RAG (coincidencia Shapley-NLI, Spearman, figuras)</li>
</ol>
<p><b>Regla de oro</b>: todas las rutas de lectura usan <code>EXPERIMENT_FINAL</code> (exp08). Nunca <code>EXPERIMENT_NAME</code> ni <code>EXPERIMENT_ID</code>.</p>


<h2>0. Guard de seguridad y configuracion</h2>
<p>Verificacion de que el experimento canonico es exp08 y de que los recursos de solo-lectura existen en disco. Esta celda debe ejecutarse antes que cualquier otra.</p>


In [1]:
## Guard de seguridad: asegura que el pipeline XAI apunta a exp08
## Esta celda debe ejecutarse ANTES que cualquier otra
import sys
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

## Rutas del proyecto
XAI_ROOT     = Path('').resolve()       ## Tesis/XAI/  (notebook esta en XAI/)
PROJECT_ROOT = XAI_ROOT.parent          ## Tesis/
SRC_DIR      = PROJECT_ROOT / 'src'
XAI_MOD_DIR  = XAI_ROOT / 'xai'        ## Tesis/XAI/xai/

## Agregar al sys.path los directorios necesarios para los modulos del proyecto
## xai/  -> permite: from config_xai import ... (igual que hacen los modulos internos)
## src/  -> permite: from models import MammoVLM
for d in [str(XAI_MOD_DIR), str(SRC_DIR)]:
    if d not in sys.path:
        sys.path.insert(0, d)

## Importar config y activar el guard de EXPERIMENT_FINAL
from config_xai import (
    EXPERIMENT_FINAL,
    EXP08_MODEL_PT, MAMMOCLIP_CHECKPOINT,
    TEST_CSV, FINDING_ANNOTATIONS_CSV,
    RAG_INDEX_DIR, LITERATURE_DIR,
    IMAGE_HEIGHT, IMAGE_WIDTH,
    OUT_BIRADS, OUT_DENSITY, OUT_RAG, OUT_FIGURAS, OUT_TABLAS,
)

## Verificar que los recursos de solo-lectura existen
assert EXP08_MODEL_PT.exists(),         f'Checkpoint exp08 no encontrado: {EXP08_MODEL_PT}'
assert MAMMOCLIP_CHECKPOINT.exists(),   f'Mammo-CLIP no encontrado: {MAMMOCLIP_CHECKPOINT}'
assert TEST_CSV.exists(),               f'Test CSV no encontrado: {TEST_CSV}'
assert FINDING_ANNOTATIONS_CSV.exists(), f'Finding annotations no encontrado: {FINDING_ANNOTATIONS_CSV}'

print(f'Experimento canonico : {EXPERIMENT_FINAL}')
print(f'Checkpoint modelo    : {EXP08_MODEL_PT}')
print(f'Resolucion entrada   : {IMAGE_HEIGHT}x{IMAGE_WIDTH}')
print('Guard OK: todos los recursos de solo-lectura existen')


Experimento canonico : exp08_ordinal_sord_qwk_descongelado
Checkpoint modelo    : /home/gtrujillod/Tesis/outputs/experiments/exp08_ordinal_sord_qwk_descongelado/model.pt
Resolucion entrada   : 1520x912
Guard OK: todos los recursos de solo-lectura existen


<h2>Paso 1a: Carga del modelo clasificador exp08</h2>
<p>Instancia <code>MammoVLM</code> con la configuracion de exp08 (EfficientNet-B5 + dual head, unfreeze_last_n_blocks=2), carga el state dict desde <code>outputs/experiments/exp08_.../model.pt</code> y prepara el modelo para atribucion (float32, modo eval, gradientes habilitados).</p>


In [2]:
from carga_modelo import cargar_modelo_exp08, cargar_transform_inferencia

model, device = cargar_modelo_exp08(device='auto')
transform      = cargar_transform_inferencia()

print(f'Dispositivo efectivo : {device}')
print(f'Parametros totales   : {sum(p.numel() for p in model.parameters()):,}')
print(f'Parametros trainables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


Dispositivo efectivo : cuda
Parametros totales   : 31,442,209
Parametros trainables: 10,057,737


<h2>Paso 1b: Carga del pipeline RAG</h2>
<p>Carga el indice FAISS desde <code>data/rag_index/</code> (solo-lectura, 6869 chunks, PubMedBERT 768-dim) y el LLM Qwen2.5-7B en bfloat16. Tambien carga el modelo NLI mDeBERTa-v3 para el grounding del Bloque B.</p>
<p><b>Nota</b>: esta celda consume ~14-16 GB de VRAM (Qwen2.5-7B en bfloat16). Ejecutar solo si se dispone de GPU con memoria suficiente o en CPU.</p>


In [3]:
from carga_rag import cargar_pipeline_rag, cargar_nli

## Cargar indice FAISS + LLM Qwen2.5-7B
generator, retriever, indexer, llm, tokenizer = cargar_pipeline_rag(device='auto')

## Cargar modelo NLI para grounding (Bloque B)
nli_model, nli_tokenizer, entailment_idx = cargar_nli(device='auto')

print(f'Chunks en el indice FAISS: {len(indexer.chunks)}')
print(f'Indice entailment NLI    : {entailment_idx}')
print(f'LLM                      : {llm.__class__.__name__} en {llm.device}')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Chunks en el indice FAISS: 6869
Indice entailment NLI    : 0
LLM                      : Qwen2ForCausalLM en cuda:0


<h2>Paso 2: Atribucion del clasificador (IG y Grad-CAM)</h2>
<p>Calcula Integrated Gradients (IG) y Grad-CAM para ambas cabezas del modelo (BI-RADS y densidad) sobre el conjunto de test. Los mapas de atribucion se guardan como <code>.npz</code> en <code>XAI/outputs/clasificador/</code>.</p>
<ul>
  <li><b>IG BI-RADS</b>: objetivo = logits[3] + logits[4] (BR4+BR5 pre-softmax).</li>
  <li><b>IG densidad</b>: objetivo = logit de la clase predicha.</li>
  <li><b>Grad-CAM</b>: capa <code>encoder.backbone._conv_head</code>, upsample bilineal a 1520x912.</li>
</ul>


In [4]:
import pandas as pd
from carga_modelo import cargar_imagen, obtener_prediccion_base
from atribucion_clasificador import calcular_atribuciones_imagen, guardar_atribucion

## Cargar el test set
test_df  = pd.read_csv(str(TEST_CSV))
n_total  = len(test_df)
print(f'Test set: {n_total} imagenes')

## Iterar sobre el test set y guardar atribuciones
## (reanudar si el archivo .npz ya existe)
n_procesadas = 0
n_omitidas   = 0

for _, fila in test_df.iterrows():
    image_id   = str(fila['image_id'])
    image_path = str(fila['image_path'])

    ## Comprobar si ya se proceso esta imagen
    ruta_birads   = OUT_BIRADS   / f'{image_id}_birads.npz'
    ruta_densidad = OUT_DENSITY  / f'{image_id}_density.npz'
    if ruta_birads.exists() and ruta_densidad.exists():
        n_omitidas += 1
        continue

    ## Cargar imagen y obtener prediccion base
    img_tensor = cargar_imagen(image_path, transform, device)
    pred       = obtener_prediccion_base(model, img_tensor)

    ## Atribucion cabeza BI-RADS
    attr_birads = calcular_atribuciones_imagen(
        model, img_tensor, head='birads', device=device
    )
    meta_birads = {
        'birads_idx':   pred['birads_idx'],
        'density_idx':  pred['density_idx'],
    }
    guardar_atribucion(attr_birads, image_id, 'birads', str(OUT_BIRADS), meta=meta_birads)

    ## Atribucion cabeza densidad
    attr_density = calcular_atribuciones_imagen(
        model, img_tensor, head='density',
        predicted_class=pred['density_idx'], device=device
    )
    guardar_atribucion(attr_density, image_id, 'density', str(OUT_DENSITY), meta=meta_birads)

    n_procesadas += 1
    if n_procesadas % 100 == 0:
        print(f'Procesadas: {n_procesadas}/{n_total}  Omitidas (ya existian): {n_omitidas}')

print(f'Paso 2 completado. Nuevas: {n_procesadas}  Ya existian: {n_omitidas}')


Test set: 4000 imagenes
Paso 2 completado. Nuevas: 0  Ya existian: 4000


<h2>Paso 3: Metricas del clasificador</h2>
<p>Evalua la calidad de los mapas de atribucion con tres metricas:</p>
<ol>
  <li><b>Deletion AUC</b>: fidelidad. AUC menor = los pixels marcados son mas relevantes.</li>
  <li><b>Pointing Game</b>: localizacion. Hit si el pixel de max atribucion cae en una caja de hallazgo. Solo para BI-RADS. Reportado por grupo (sospechoso BR4-5 vs benigno).</li>
  <li><b>IoU IG-GradCAM</b>: coherencia entre los dos metodos (top-25% de pixels).</li>
</ol>


In [5]:
from atribucion_clasificador import cargar_atribucion
from metricas_clasificador import (
    preparar_cajas_test,
    evaluar_deletion_auc, evaluar_iou_ig_gradcam,
    guardar_metricas_csv,
)
from metricas_rag import evaluar_pointing_game_estratificado

## -------------------------------------------------------
## Bounding boxes escaladas a 1520x912
## -------------------------------------------------------
cajas_df = preparar_cajas_test(
    test_csv_path=str(TEST_CSV),
    findings_csv_path=str(FINDING_ANNOTATIONS_CSV),
)
print(f'Cajas: {len(cajas_df)} hallazgos en {cajas_df["image_id"].nunique()} imagenes del test')

## -------------------------------------------------------
## Deletion AUC -- cabeza BI-RADS
## IG, Grad-CAM y baseline aleatorio (referencia nula)
## -------------------------------------------------------
print('\nCalculando Deletion AUC (birads)...')
df_dauc_birads = evaluar_deletion_auc(
    model, test_df, cargar_atribucion, head='birads',
    device=device, n_sample=50, include_random_baseline=True,
)
guardar_metricas_csv(df_dauc_birads, 'deletion_auc_birads', OUT_TABLAS)

pivot_birads = (
    df_dauc_birads
    .groupby('method')['auc']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'auc_medio', 'std': 'auc_std'})
    .loc[['ig', 'gradcam', 'random']]   ## orden canonico
    .reset_index()
    .rename(columns={'method': 'metodo'})
)
print('Deletion AUC -- cabeza BI-RADS (menor = mas fiel):')
print(pivot_birads.to_string(index=False))

## -------------------------------------------------------
## Deletion AUC -- cabeza densidad
## Las atribuciones *_density.npz ya estan en disco (celda 8)
## -------------------------------------------------------
print('\nCalculando Deletion AUC (density)...')
df_dauc_density = evaluar_deletion_auc(
    model, test_df, cargar_atribucion, head='density',
    device=device, n_sample=50, include_random_baseline=True,
)
guardar_metricas_csv(df_dauc_density, 'deletion_auc_density', OUT_TABLAS)

pivot_density = (
    df_dauc_density
    .groupby('method')['auc']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'auc_medio', 'std': 'auc_std'})
    .loc[['ig', 'gradcam', 'random']]
    .reset_index()
    .rename(columns={'method': 'metodo'})
)
print('Deletion AUC -- cabeza Densidad (menor = mas fiel):')
print(pivot_density.to_string(index=False))

## -------------------------------------------------------
## Pointing Game estratificado -- cabeza BI-RADS
## IG y Grad-CAM x sospechoso/benigno
##
## Nota: la suma de imagenes por estrato puede superar el total de imagenes
## unicas con cajas (187 sospechosas + 181 benignas = 368 > 357 unicas).
## Hay imagenes con hallazgos de ambas categorias que caen en ambos grupos;
## eso es correcto: se mide si la atencion cae en el hallazgo del grupo evaluado.
## -------------------------------------------------------
print('\nCalculando Pointing Game estratificado (IG y Grad-CAM)...')
df_pointing = evaluar_pointing_game_estratificado(cargar_atribucion, cajas_df)
guardar_metricas_csv(df_pointing, 'pointing_game_estratificado', OUT_TABLAS)
print('Pointing Game -- BI-RADS (hit rate por metodo y estrato):')
print(df_pointing.to_string(index=False))

## -------------------------------------------------------
## IoU IG vs Grad-CAM -- cabeza BI-RADS (no tocar: 0.2975 ya calculado)
## -------------------------------------------------------
print('\nCalculando IoU IG-GradCAM (birads)...')
image_ids = test_df['image_id'].astype(str).tolist()
df_iou_birads = evaluar_iou_ig_gradcam(
    cargar_atribucion, image_ids, top_k=0.25, head='birads',
)
guardar_metricas_csv(df_iou_birads, 'iou_ig_gradcam_birads', OUT_TABLAS)
print(f'IoU IG-GradCAM medio BI-RADS  (top-25%): {df_iou_birads["iou"].mean():.4f}')

## -------------------------------------------------------
## IoU IG vs Grad-CAM -- cabeza densidad
## -------------------------------------------------------
print('\nCalculando IoU IG-GradCAM (density)...')
df_iou_density = evaluar_iou_ig_gradcam(
    cargar_atribucion, image_ids, top_k=0.25, head='density',
)
guardar_metricas_csv(df_iou_density, 'iou_ig_gradcam_density', OUT_TABLAS)
print(f'IoU IG-GradCAM medio Densidad (top-25%): {df_iou_density["iou"].mean():.4f}')


Cajas: 452 hallazgos en 357 imagenes del test

Calculando Deletion AUC (birads)...


Deletion AUC (birads): 100%|██████████| 50/50 [01:15<00:00,  1.50s/it]


Deletion AUC -- cabeza BI-RADS (menor = mas fiel):
 metodo  auc_medio  auc_std
     ig   0.146918 0.033670
gradcam   0.385118 0.143620
 random   0.080172 0.020729

Calculando Deletion AUC (density)...


Deletion AUC (density): 100%|██████████| 50/50 [01:15<00:00,  1.50s/it]


Deletion AUC -- cabeza Densidad (menor = mas fiel):
 metodo  auc_medio  auc_std
     ig   0.222576 0.072999
gradcam   0.412989 0.247781
 random   0.472161 0.209874

Calculando Pointing Game estratificado (IG y Grad-CAM)...
cajas_df columnas: ['study_id', 'series_id', 'image_id', 'laterality', 'view_position', 'height', 'width', 'breast_birads', 'breast_density', 'finding_categories', 'finding_birads', 'xmin', 'ymin', 'xmax', 'ymax', 'split', 'image_path', 'escala_x', 'escala_y', 'xmin_s', 'ymin_s', 'xmax_s', 'ymax_s']
finding_birads  dtype=Int64  unique=[np.int64(3), np.int64(4), np.int64(5)]  NaN=23
image_ids en cajas_df: 357  con .npz en OUT_BIRADS: 357
Grupo 'sospechoso': 187 imagenes unicas


...

Pointing Game -- BI-RADS (hit rate por metodo y estrato):
 metodo      grupo  hit_rate  n_imagenes
     ig sospechoso  0.518717         187
gradcam sospechoso  0.374332         187
     ig    benigno  0.381215         181
gradcam    benigno  0.215470         181

Calculando IoU IG-GradCAM (birads)...


IoU IG vs GradCAM: 100%|██████████| 4000/4000 [04:49<00:00, 13.82it/s]


IoU IG-GradCAM medio BI-RADS  (top-25%): 0.2975

Calculando IoU IG-GradCAM (density)...


IoU IG vs GradCAM: 100%|██████████| 4000/4000 [04:53<00:00, 13.64it/s]

IoU IG-GradCAM medio Densidad (top-25%): 0.1874


<h2>Paso 3b: Visualizacion cualitativa de las atribuciones</h2>
<p>Seleccion determinista (seed=42) de 5 imagenes representativas para validar
visualmente los mapas de atribucion sobre el mamograma original:</p>
<ol>
  <li><b>Sospechoso HIT 1</b>: caso BR4/5 donde el pixel de maxima atribucion (IG) cae dentro de la caja GT.</li>
  <li><b>Sospechoso HIT 2</b>: segundo caso exitoso como referencia adicional.</li>
  <li><b>Sospechoso MISS</b>: modo de falla; el modelo detecta malignidad pero senala otra region.</li>
  <li><b>Benigno BR3</b>: como atiende el modelo a hallazgos no malignos.</li>
  <li><b>Mayor divergencia IG-GradCAM</b>: imagen con menor IoU entre los dos metodos (desde df_iou_birads); ilustra casos donde los metodos discrepan.</li>
</ol>
<p>Para cada imagen se generan 3 paneles: original en gris (cajas GT en verde),
IG con alpha-blend (hot, 0.4) y Grad-CAM con alpha-blend. La cruz roja marca
el pixel de maxima atribucion; el titulo indica HIT o MISS del Pointing Game.</p>
<p>Salidas en <code>XAI/outputs/figuras/</code>:</p>
<ul>
  <li><code>overlay_atribuciones.png</code>: grid 5x3, 300 dpi.</li>
  <li><code>dual_head_contraste.png</code>: 2x3, contraste BI-RADS (focal) vs Densidad (difuso).</li>
</ul>
<p>Los 5 <code>image_id</code> seleccionados se guardan en
<code>XAI/outputs/tablas/seleccion_visualizacion.csv</code> para trazabilidad.</p>


In [7]:
import numpy as np
import pandas as pd
from metricas_clasificador import pointing_game_imagen
from atribucion_clasificador import cargar_atribucion
from visualizacion_xai import figura_grid_seleccion, figura_dual_head

## -------------------------------------------------------
## Calcular hit IG por imagen sospechosa para la seleccion
## cajas_df y df_iou_birads ya estan definidos en la celda anterior
## -------------------------------------------------------
birads_numerico_cajas = (
    cajas_df['finding_birads']
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .astype('Int64')
)

sosp_df = cajas_df[birads_numerico_cajas.isin([4, 5])].copy()
br3_df  = cajas_df[birads_numerico_cajas == 3].copy()

sosp_hits   = []
sosp_misses = []
for img_id in sorted(sosp_df['image_id'].unique()):   ## sorted: orden determinista
    img_cajas = sosp_df[sosp_df['image_id'] == img_id]
    try:
        attr = cargar_atribucion(img_id, 'birads', str(OUT_BIRADS))
    except FileNotFoundError:
        continue
    if pointing_game_imagen(attr['ig'], img_cajas):
        sosp_hits.append(img_id)
    else:
        sosp_misses.append(img_id)

br3_ids_con_attr = [
    i for i in sorted(br3_df['image_id'].unique())
    if (OUT_BIRADS / f'{i}_birads.npz').exists()
]

print(f'Sospechosos con HIT IG: {len(sosp_hits)}')
print(f'Sospechosos con MISS IG: {len(sosp_misses)}')
print(f'Benignos BR3 con atribucion: {len(br3_ids_con_attr)}')

## -------------------------------------------------------
## Seleccion determinista con seed=42
## -------------------------------------------------------
rng_sel = np.random.default_rng(42)

def _pick(lista, n):
    ## lista ya ordenada al construirla; choice es reproducible con seed fijo
    n = min(n, len(lista))
    if n == 0:
        return []
    idx = rng_sel.choice(len(lista), size=n, replace=False)
    return [lista[i] for i in sorted(idx)]

sel_hits = _pick(sosp_hits, 2)    ## 2 sospechosos con HIT
sel_miss = _pick(sosp_misses, 1)  ## 1 sospechoso con MISS
sel_br3  = _pick(br3_ids_con_attr, 1)  ## 1 benigno BR3

if len(sel_hits) < 2:
    raise RuntimeError(f'Insuficientes HITs sospechosos ({len(sel_hits)}). Revisar atribuciones.')
if not sel_miss:
    raise RuntimeError('Sin MISSes sospechosos para mostrar.')
if not sel_br3:
    raise RuntimeError('Sin casos BR3 con atribucion disponible.')

## Quinta imagen: menor IoU IG-GradCAM entre imagenes con cajas (max divergencia)
## df_iou_birads viene de la celda anterior (Paso 3)
ids_con_cajas = set(cajas_df['image_id'].unique())
excluidos     = set(sel_hits + sel_miss + sel_br3)
iou_candidatos = (
    df_iou_birads[df_iou_birads['image_id'].isin(ids_con_cajas - excluidos)]
    .sort_values('iou')
    .reset_index(drop=True)
)
if not iou_candidatos.empty:
    sel_div = iou_candidatos.loc[0, 'image_id']
else:
    ## Fallback: segundo benigno BR3 si no hay candidatos de IoU
    fallback = [i for i in br3_ids_con_attr if i not in excluidos]
    sel_div  = _pick(fallback, 1)[0] if fallback else sel_br3[0]

## Lista final: (image_id, etiqueta)
seleccion_ids_labels = [
    (sel_hits[0], 'sospechoso HIT 1'),
    (sel_hits[1], 'sospechoso HIT 2'),
    (sel_miss[0], 'sospechoso MISS'),
    (sel_br3[0],  'benigno BR3'),
    (sel_div,     'mayor divergencia IG-GradCAM'),
]

print('\nImagenes seleccionadas para visualizacion cualitativa:')
for img_id, etiqueta in seleccion_ids_labels:
    print(f'  {etiqueta:35s}: {img_id}')

## Guardar para trazabilidad
df_sel = pd.DataFrame(seleccion_ids_labels, columns=['image_id', 'etiqueta'])
OUT_TABLAS.mkdir(parents=True, exist_ok=True)
df_sel.to_csv(str(OUT_TABLAS / 'seleccion_visualizacion.csv'), index=False)

## -------------------------------------------------------
## Construir lista de dicts para figura_grid_seleccion
## Se usan TODAS las cajas de cajas_df para la imagen (no solo el estrato)
## -------------------------------------------------------
def _img_path(img_id):
    fila = test_df[test_df['image_id'] == img_id]
    return str(fila['image_path'].iloc[0]) if not fila.empty else None

seleccion = []
for img_id, label in seleccion_ids_labels:
    seleccion.append({
        'image_id':     img_id,
        'image_path':   _img_path(img_id),
        'cajas_imagen': cajas_df[cajas_df['image_id'] == img_id],
        'label':        label,
    })

## -------------------------------------------------------
## Figura principal: grid 5 x 3 (300 dpi)
## -------------------------------------------------------
print('\nGenerando grid 5x3 de overlays...')
ruta_grid = figura_grid_seleccion(
    seleccion, cargar_atribucion, transform, device,
    head='birads', out_figuras_dir=OUT_FIGURAS,
    nombre='overlay_atribuciones',
)
print(f'Grid guardado: {ruta_grid}')

## -------------------------------------------------------
## Figura opcional: contraste dual-head (birads vs density)
## Imagen: primer sospechoso HIT (lesion clara para la comparacion)
## -------------------------------------------------------
print('\nGenerando contraste dual-head (birads focal vs density difuso)...')
img_dual = sel_hits[0]
ruta_dual = figura_dual_head(
    image_id=img_dual,
    image_path=_img_path(img_dual),
    cajas_df_imagen=cajas_df[cajas_df['image_id'] == img_dual],
    attr_load_func=cargar_atribucion,
    transform=transform,
    device=device,
    out_figuras_dir=OUT_FIGURAS,
    nombre='dual_head_contraste',
)
print(f'Contraste dual-head guardado: {ruta_dual}')


Sospechosos con HIT IG: 97
Sospechosos con MISS IG: 90
Benignos BR3 con atribucion: 181

Imagenes seleccionadas para visualizacion cualitativa:
  sospechoso HIT 1                   : 11dbf2405e94335db8d8011db532a5ac
  sospechoso HIT 2                   : b68769ba266c8a6ccfc95b649e14e23c
  sospechoso MISS                    : 6904df2bf7180487ad11bf6da7890b5b
  benigno BR3                        : 73dcab1172639ae173f533e072daf2a4
  mayor divergencia IG-GradCAM       : 0e31855d02eadf8670ffaeeaeddbf229

Generando grid 5x3 de overlays...
Grid guardado: /home/gtrujillod/Tesis/XAI/outputs/figuras/overlay_atribuciones.png

Generando contraste dual-head (birads focal vs density difuso)...
Contraste dual-head guardado: /home/gtrujillod/Tesis/XAI/outputs/figuras/dual_head_contraste.png


<h2>Paso 4: Atribucion del RAG (Shapley + NLI)</h2>
<p>Para cada imagen de una muestra estratificada (max 40 por clase BI-RADS de referencia, sin reemplazo):</p>
<ol>
  <li>Se genera el informe UNA VEZ con decodificacion greedy (determinista, <code>do_sample=False</code>).</li>
  <li>Se evaluan los 8 subconjuntos de chunks (2<sup>3</sup>) con teacher forcing sobre ese informe fijo. Los IDs de tokens se concatenan (no los textos) para evitar fusion BPE.</li>
  <li>Se verifica la propiedad de eficiencia de Shapley: sum(phi_j) = v(full) - v(empty).</li>
  <li>Se calcula el grounding NLI: P(entailment | chunk_j, oracion_s).</li>
</ol>
<p>Flujo: <b>4a</b> = limpieza de outputs previos, <b>4b</b> = validacion sobre 3 casos, <b>4c</b> = loop completo.</p>


<h3>Paso 4 (inicio): Definicion de la muestra RAG</h3>
<p>La muestra estratificada se define aqui, una sola vez, para que las tres
secciones siguientes (4a limpieza, 4b validacion, 4c loop) la reutilicen
sin riesgo de referencias hacia adelante.</p>
<p>Estrategia: max 40 imagenes por clase BI-RADS de referencia, sin reemplazo.
Los <code>image_id</code> se guardan para trazabilidad.</p>


In [6]:
from metricas_rag import crear_muestra_rag

## Definir la muestra una sola vez; 4b, 4c y Paso 5 la reutilizan
muestra_df = crear_muestra_rag(test_df=test_df, n_por_clase=40, seed=42)

print(f'Muestra RAG: {len(muestra_df)} imagenes')
print(muestra_df.groupby('birads_clase').size().to_frame('n').to_string())

## Guardar image_ids para trazabilidad de la corrida
OUT_TABLAS.mkdir(parents=True, exist_ok=True)
muestra_df[['image_id', 'birads_clase']].to_csv(
    str(OUT_TABLAS / 'muestra_rag_image_ids.csv'), index=False
)
print('image_ids guardados en OUT_TABLAS/muestra_rag_image_ids.csv')


Muestra RAG: 160 imagenes
               n
birads_clase    
1             40
2             40
3             40
4             40
image_ids guardados en OUT_TABLAS/muestra_rag_image_ids.csv


<h2>4a. Limpieza de outputs RAG anteriores</h2>
<p>Antes de re-correr el Paso 4, eliminar los JSON de ejecuciones previas para que el guard de idempotencia no conserve cifras con bugs previos. Ejecutar solo si se desea una corrida limpia.</p>


In [ ]:
## Limpieza de outputs RAG de ejecuciones anteriores.
## Por defecto False para no destruir datos por accidente.
## Poner True para borrar todos los *_rag.json de OUT_RAG antes de re-correr.
CONFIRMAR_LIMPIEZA_RAG = False

archivos_rag = list(OUT_RAG.glob('*_rag.json'))
if not CONFIRMAR_LIMPIEZA_RAG:
    print(f'Modo seguro (CONFIRMAR_LIMPIEZA_RAG=False).')
    print(f'  Archivos *_rag.json encontrados en OUT_RAG: {len(archivos_rag)}')
    print('  Para borrarlos, poner CONFIRMAR_LIMPIEZA_RAG = True y re-ejecutar.')
elif archivos_rag:
    for archivo in archivos_rag:
        archivo.unlink()
    print(f'{len(archivos_rag)} archivos *_rag.json eliminados de {OUT_RAG}.')
else:
    print('No habia archivos RAG previos. Directorio ya limpio.')


<h3>4b. Validacion sobre 3 casos antes del loop completo</h3>
<p>Verifica las 3 aserciones:</p>
<ol>
  <li><b>Eficiencia de Shapley</b>: sum(phi_j) = v(full) - v(empty).</li>
  <li><b>Alineacion teacher forcing</b>: bajo el subconjunto completo, el argmax por token reproduce el target greedy.</li>
  <li><b>Mapeo BI-RADS</b>: el informe declara la categoria = indice_predicho + 1.</li>
</ol>


In [ ]:
import re
import torch
from carga_rag import prediccion_a_dict_generador
from atribucion_rag import (
    generar_informe_greedy, calcular_shapley_rag,
    construir_prompt_con_subconjunto,
)

## -------------------------------------------------------
## Confirmacion del shift logit->token antes del loop:
##   El modelo autoregresivo produce logits[t] que predicen token[t+1].
##   Para evaluar P(target_ids[0] | prefix), usamos logits[prefix_len - 1].
##   En general: logits[prefix_len + i - 1] predice target_ids[i].
##   Slice correcto: logits[prefix_len-1 : prefix_len+target_len-1]
##                               corresponde a tokens: target_ids[0:target_len]
## La tasa argmax==target se mide EXCLUSIVAMENTE bajo el subconjunto COMPLETO
## (los 3 chunks), que es el mismo contexto bajo el que el target fue generado
## con greedy. Medir bajo un subconjunto parcial baja la tasa por diseno
## (menor contexto = menor probabilidad del texto especifico), no por bug.
## -------------------------------------------------------

casos_val = muestra_df.head(3)
tasas_por_caso = []

for i, (_, fila) in enumerate(casos_val.iterrows()):
    image_id   = str(fila['image_id'])
    image_path = str(fila['image_path'])
    print(f'\n=== Caso {i+1}: {image_id} ===')

    img_tensor = cargar_imagen(image_path, transform, device)
    pred_base  = obtener_prediccion_base(model, img_tensor)
    pred_dict  = prediccion_a_dict_generador(pred_base)
    birads_esperado = pred_dict['birads_pred'] + 1
    print(f'  BI-RADS predicho (indice={pred_dict["birads_pred"]}) => nivel clinico {birads_esperado}')

    ## Generar informe greedy (target fijo, contexto = subconjunto COMPLETO)
    informe, chunks = generar_informe_greedy(generator, pred_dict, retriever)
    print(f'  Informe greedy: {len(informe)} chars, {len(chunks)} chunks')

    ## Asercin 3: mapeo BI-RADS en el informe
    patron = rf'BI.?RADS\s*{birads_esperado}'
    match  = bool(re.search(patron, informe, re.IGNORECASE))
    print(f'  [A3] Informe menciona BI-RADS {birads_esperado}: {match}'
          + ('' if match else f'  <- primeras 200 chars: {informe[:200]}'))

    ## Eficiencia Shapley
    shapley = calcular_shapley_rag(
        generator=generator, llm=llm, tokenizer=tokenizer,
        prediction_dict=pred_dict,
        chunks_recuperados=chunks, informe_objetivo=informe,
    )
    print(f'  [A1] Eficiencia Shapley: error={shapley["eficiencia_error"]:.2e}')

    ## Alineacion teacher forcing bajo subconjunto COMPLETO (chunks [0,1,2])
    ## Razon: el target se genero con los 3 chunks; medir con menos contexto
    ## reduciria la tasa por diseno y no indicaria bug.
    prompt_completo = construir_prompt_con_subconjunto(
        generator, pred_dict, chunks, [0, 1, 2]   ## <- subconjunto COMPLETO
    )

    prefix_ids = tokenizer.encode(prompt_completo, add_special_tokens=False)
    target_ids = tokenizer.encode(informe,         add_special_tokens=False)
    ## Concatenar IDs (no textos) para evitar fusion BPE en la frontera
    full_ids   = prefix_ids + target_ids
    P = len(prefix_ids)
    T = len(target_ids)

    with torch.no_grad():
        logits = llm(
            input_ids=torch.tensor([full_ids], device=llm.device)
        ).logits[0]   ## [P+T, vocab_size]

    ## Shift: logits[t] predice token[t+1]
    ## Para target: logits[P-1 .. P+T-2] predicen target_ids[0 .. T-1]
    logits_target  = logits[P - 1 : P + T - 1]        ## [T, vocab_size]
    argmax_target  = logits_target.argmax(dim=-1)      ## [T]
    target_t       = torch.tensor(target_ids, device=llm.device)

    coincidencias  = (argmax_target == target_t)
    tasa_match     = coincidencias.float().mean().item()
    tasas_por_caso.append(tasa_match)

    ## Mostrar los primeros 5 tokens del target para confirmar el shift visualmente
    muestra_tokens = []
    for k_tok in range(min(5, T)):
        tok_real = target_ids[k_tok]
        tok_pred = argmax_target[k_tok].item()
        coincide = 'OK' if tok_real == tok_pred else 'X'
        muestra_tokens.append(
            f'{coincide} [{tokenizer.decode([tok_real])!r}->{tokenizer.decode([tok_pred])!r}]'
        )
    print(f'  [A2] Contexto: subconjunto COMPLETO (3 chunks)')
    print(f'       Shift: logits[P-1..P+T-2] => target_ids[0..T-1]  (P={P}, T={T})')
    print(f'       Primeros 5 tokens: {"  ".join(muestra_tokens)}')
    print(f'       Tasa argmax==target: {tasa_match:.4f}  ({coincidencias.sum().item()}/{T} tokens)')
    if tasa_match < 0.90:
        print('       ADVERTENCIA: tasa < 0.90. Revisar implementacion del shift.')

## Resumen de los 3 casos
import statistics
media_tasa = statistics.mean(tasas_por_caso)
print(f'\n--- Resumen de alineacion teacher forcing (3 casos) ---')
for j, t in enumerate(tasas_por_caso, 1):
    print(f'  Caso {j}: {t:.4f}')
print(f'  Media : {media_tasa:.4f}')
print('\nValidacion 4b completada.')


<h3>4c. Loop completo sobre la muestra</h3>


In [ ]:
from carga_rag import prediccion_a_dict_generador
from atribucion_rag import (
    calcular_atribuciones_rag, guardar_atribucion_rag
)

## muestra_df ya esta definida en la celda de inicio del Paso 4
n_procesadas_rag = 0
n_omitidas_rag   = 0

for _, fila in muestra_df.iterrows():
    image_id   = str(fila['image_id'])
    image_path = str(fila['image_path'])

    ## Guard de idempotencia: saltar si ya se proceso
    if (OUT_RAG / f'{image_id}_rag.json').exists():
        n_omitidas_rag += 1
        continue

    img_tensor = cargar_imagen(image_path, transform, device)
    pred_base  = obtener_prediccion_base(model, img_tensor)
    pred_dict  = prediccion_a_dict_generador(pred_base)

    ## calcular_atribuciones_rag verifica eficiencia de Shapley internamente
    resultado = calcular_atribuciones_rag(
        generator=generator, retriever=retriever,
        llm=llm, tokenizer=tokenizer,
        nli_model=nli_model, nli_tokenizer=nli_tokenizer,
        entailment_idx=entailment_idx,
        prediction_dict=pred_dict,
    )
    guardar_atribucion_rag(resultado, image_id)
    n_procesadas_rag += 1

    if n_procesadas_rag % 10 == 0:
        print(f'RAG procesadas: {n_procesadas_rag}  Omitidas: {n_omitidas_rag}')

print(f'Paso 4c completado. Nuevas: {n_procesadas_rag}  Ya existian: {n_omitidas_rag}')


<h2>Paso 5: Metricas del RAG</h2>
<p>Compara la atribucion Shapley con el grounding NLI para medir coherencia:</p>
<ul>
  <li><b>Tasa de coincidencia top-1</b>: fraccion de casos en que el chunk con mayor valor Shapley es tambien el de mayor entailment NLI.</li>
  <li><b>Correlacion de Spearman</b>: entre el ranking Shapley y el ranking NLI por imagen.</li>
  <li><b>Score de grounding</b>: entailment NLI del chunk mas influyente segun Shapley.</li>
</ul>
<p>Los resultados se guardan en <code>XAI/outputs/tablas/</code> y <code>XAI/outputs/figuras/</code>.</p>


In [ ]:
from metricas_rag import (
    evaluar_muestra_rag, calcular_resumen_rag,
    guardar_metricas_rag_csv, figura_distribucion_shapley,
)

## Evaluar metricas sobre todas las imagenes RAG procesadas
image_ids_rag = muestra_df['image_id'].astype(str).unique().tolist()

df_metricas_rag = evaluar_muestra_rag(image_ids_rag)
print(f'Imagenes evaluadas: {len(df_metricas_rag)}')

## Resumen POR CLASE (no pooled — la muestra es estratificada)
## calcular_resumen_rag retorna (df_por_clase, df_total)
df_por_clase, df_total = calcular_resumen_rag(df_metricas_rag)

print('\n--- Tasa de coincidencia Shapley-NLI POR CLASE BI-RADS ---')
print(df_por_clase.to_string(index=False))

print('\n--- Macro-promedio (promedio de clases, no pooled) ---')
print(df_total.to_string(index=False))

## Guardar CSVs
guardar_metricas_rag_csv(df_metricas_rag, 'metricas_rag_por_imagen')
guardar_metricas_rag_csv(df_por_clase,    'metricas_rag_por_clase')
guardar_metricas_rag_csv(df_total,        'metricas_rag_resumen')

## Figura: distribucion NLI del top-Shapley y coincidencia por clase
figura_distribucion_shapley(df_metricas_rag)

print('Paso 5 completado. Tablas y figura guardadas.')
